# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fatima12aa/fa-ml/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [2]:
import os
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
os.environ["HF_TOKEN"] = hf_token
print("Token loaded:", hf_token[:8] + "..." if hf_token else "NOT FOUND")

Token loaded: hf_DCVCv...


In [3]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
print("Tables defined.")

Tables defined.


In [4]:
con.sql(f"SELECT * FROM {TABLES['dim_clients']} LIMIT 5").show()

┌─────────────────────────┬───────────┬────────────────┬────────────────┬───────────────────────────────┬─────────────────────┬─────────────────────┬────────────────┬────────────────┐
│     client_hash_id      │ is_active │ has_gsc_access │ has_ga4_access │        access_profile         │ client_created_date │ client_updated_date │ gsc_data_start │ ga4_data_start │
│         varchar         │  boolean  │    boolean     │    boolean     │            varchar            │        date         │        date         │      date      │      date      │
├─────────────────────────┼───────────┼────────────────┼────────────────┼───────────────────────────────┼─────────────────────┼─────────────────────┼────────────────┼────────────────┤
│ client_04660893ae39614a │ true      │ true           │ true           │ gsc_and_ga4                   │ 2026-04-15          │ 2026-06-27          │ NULL           │ 2026-05-22     │
│ client_05475c07ed21a83a │ true      │ false          │ false          │ no_sea

In [5]:
con.sql(f"SELECT * FROM {TABLES['dim_content']} LIMIT 5").show()

┌─────────────────────────┬──────────────────────────┬──────────────────────────┬──────────────────────┬────────────────────┬─────────────────────┬────────────────┬──────────────────────┬──────────────────────┬─────────────────┬───────────────┬─────────────┬───────────────────┬────────┬───────────────┬───────────┬────────────────┬──────────────────────┬─────────────────────────┬────────────────────────┬────────────┬────────────┬─────────────────────┬────────────────────────────┬──────────────┬────────────┐
│     client_hash_id      │     content_hash_id      │     keyword_hash_id      │     url_hash_id      │ keyword_char_count │ keyword_token_count │ url_char_count │ content_created_date │ content_updated_date │  content_type   │ search_volume │ competition │ competition_level │  cpc   │  main_intent  │ backlinks │ category_count │ keyword_created_date │      provider_used      │       model_used       │ char_count │ word_count │ last_optimized_date │ optimization_eligible_date │ is_p

In [6]:
#i mistakenly used the csv file first so all the code blocks have my answres with refernce to the csv first and then with reference to the huggingface dataset
import pandas as pd
df = pd.read_csv("content_refresh_anonymized.csv")
df.columns.tolist()
#one row = one page. There are no explicit dates/time frames provided in the csv file. Instead the data available is relative (since 90 days or 30 days) as is seen by these columns



['content_id',
 'client_id',
 'search_volume',
 'competition',
 'competition_level',
 'cpc',
 'content_type',
 'main_intent',
 'word_count',
 'char_count',
 'provider_used',
 'model_used',
 'impressions_90d',
 'clicks_90d',
 'pageviews_90d',
 'sessions_90d',
 'users_90d',
 'engaged_sessions_90d',
 'ai_sessions_90d',
 'scroll_events_90d',
 'days_with_impressions',
 'days_with_sessions',
 'impressions_last_30d',
 'clicks_last_30d',
 'sessions_last_30d',
 'impressions_prev_30d',
 'clicks_prev_30d',
 'sessions_prev_30d',
 'content_age_days',
 'age_tier',
 'age_tier_order',
 'days_since_last_update',
 'freshness_tier',
 'word_count_tier',
 'char_count_tier',
 'ctr',
 'avg_position',
 'engagement_rate',
 'scroll_rate',
 'ai_traffic_pct',
 'impression_tier',
 'position_tier',
 'trend_direction',
 'trend_pct']

In [7]:
print((df["impressions_last_30d"] == df["impressions_prev_30d"]).mean())
#however, it is important to note the ovelap between the entries in impressions_last_30d and impressions_prev_30d is only around 5.3 percent. this shows that both these columns are not duplicates and hence the data available was recorded at different times. These are genuinely separate windows, not just relabeled duplicates.

0.05313333333333333


1. Unit of analysis + time window: One row = one what, over which dates?

One row = one content item (page) on one specific date. This is the grain of fact_content_daily_performance — daily × client × content — unlike the starter CSV, where each row was a pre-aggregated 90-day summary per page. Our verification and feature-building slice uses month=2026-03 (a mid-panel month with real data before and after it), while our label compares March against February.
3. Time window:

month=2026-03 for verification queries and feature-building, deliberately not the final month (June 2026) — using the last month would risk leakage, since there'd be no true "future" data left to validate against, and the last month is the natural outcome window for any past→future label.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [8]:
df["computed_pct_check"] = (df["impressions_last_30d"] - df["impressions_prev_30d"]) / df["impressions_prev_30d"] * 100
df[["trend_pct", "computed_pct_check"]].head(10)
df["provider_used"].unique()
df["model_used"].unique()
df["provider_used"].isna().mean()
df["ai_traffic_pct"].isna().mean()
(df["ai_traffic_pct"] == 0).mean()

np.float64(0.9356666666666666)

Feature — real, observed signals knowable before the outcome: content_age_days, days_since_last_update, search_volume, competition, cpc, word_count, char_count, impressions_90d, clicks_90d, pageviews_90d, sessions_90d, users_90d, engaged_sessions_90d, scroll_events_90d, days_with_impressions, days_with_sessions, impressions_last_30d/prev_30d, clicks_last_30d/prev_30d, sessions_last_30d/prev_30d, ctr, avg_position, engagement_rate, scroll_rate.

Label — trend_direction (the source of is_declining_label, our proxy target).

Context — content_id, client_id (pseudonymized identifiers, no predictive pattern); age_tier, freshness_tier, word_count_tier, char_count_tier, impression_tier, position_tier (bucketed duplicates of raw numeric columns already used as features — redundant, kept only for human-readable reason codes).

Excluded —

trend_pct: verified numerically (via (last_30d - prev_30d)/prev_30d * 100) to be the exact value trend_direction is derived from. Using it would be leakage — the model would learn to copy the label instead of finding real signal.
provider_used, model_used: 71.5% missing, and even where present, risk acting as a confound — any correlation between AI model and decline may really reflect when that model was adopted, not genuine content quality.
ai_traffic_pct: no missing values, but 93.56% of rows are exactly 0, meaning very little real signal for the vast majority of pages — consistent with the lane guide's warning that AI-session data is generally sparse. Excluded rather than treated as a normal feature.

from hugging face dataset:
2. Fields: which table(s)?

fact_content_daily_performance (or fact_daily_sample for quick testing) for the daily traffic/click/position signals, joined to dim_content for static page-level attributes (word_count, content_type, search_volume, etc.) via content_hash_id. dim_clients may also be used for the availability check (has_gsc_access, has_ga4_access boolean flags).

4. Target or proxy:

A proxy label — "declining" — defined by comparing a page's traffic (e.g. impressions) in March 2026 against February 2026. If March impressions are meaningfully lower than February's (e.g. a chosen % drop threshold), the page is labeled declining (1), otherwise not declining (0). This mirrors trend_direction from the starter CSV, but computed ourselves from two real, verifiable months of daily data.

5. What we deliberately exclude:

Any data from March itself (or later) must never be used as a feature — only as the label — since March is the outcome we're trying to predict. Including March's own traffic numbers as a feature would be leakage, the same trap as trend_pct in the starter CSV, just reshaped around real calendar time instead of a pre-built column.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [9]:
# Claim 1: one row = one page (grain check)
print("Shape:", df.shape)
print("Unique content_id count:", df["content_id"].nunique())
# If nunique == number of rows, confirms one row per page, no duplicates

# Claim 2: last_30d and prev_30d are genuinely separate windows, not duplicates
overlap_rate = (df["impressions_last_30d"] == df["impressions_prev_30d"]).mean()
print("Fraction identical between last_30d and prev_30d:", overlap_rate)
# ~5.3% identical -> genuinely separate windows, not a duplicated column

# Claim 3: trend_pct is derived from last_30d/prev_30d, and trend_direction from trend_pct -> leakage
df["computed_pct_check"] = (df["impressions_last_30d"] - df["impressions_prev_30d"]) / df["impressions_prev_30d"] * 100
print(df[["trend_pct", "computed_pct_check"]].head(10))
# Near-identical values confirm trend_pct's formula, and therefore why it must be excluded

# Claim 4: provider_used / model_used are heavily missing
print("provider_used missing rate:", df["provider_used"].isna().mean())
print("model_used unique values:", df["model_used"].unique())

# Claim 5: ai_traffic_pct has no NaNs but is mostly zero
print("ai_traffic_pct missing rate:", df["ai_traffic_pct"].isna().mean())
print("ai_traffic_pct == 0 rate:", (df["ai_traffic_pct"] == 0).mean())

# Claim 6: avg_position uses 0 as a sentinel for "no data," distorting the mean if left in
print("Rows with avg_position == 0:", (df["avg_position"] == 0).sum())
print("Mean avg_position including zeros:", df["avg_position"].mean())
print("Mean avg_position excluding zeros:", df[df["avg_position"] > 0]["avg_position"].mean())

Shape: (30000, 45)
Unique content_id count: 30000
Fraction identical between last_30d and prev_30d: 0.05313333333333333
   trend_pct  computed_pct_check
0      -41.4          -41.438703
1      -57.7          -57.717667
2      -60.9          -60.880276
3      -13.8          -13.789824
4      -34.7          -34.733416
5      -38.9          -38.850347
6      -92.3          -92.307692
7        0.6            0.632911
8      -58.8          -58.808215
9      -29.2          -29.213483
provider_used missing rate: 0.7146
model_used unique values: ['gemini-2.5-flash' 'gemini-3-flash-preview' nan 'gpt-4o-mini' 'unknown'
 'gpt-5-mini']
ai_traffic_pct missing rate: 0.0
ai_traffic_pct == 0 rate: 0.9356666666666666
Rows with avg_position == 0: 1205
Mean avg_position including zeros: 16.342380000000002
Mean avg_position excluding zeros: 17.026268449383576


In [10]:
# Claim: one row = one page (content_hash_id) on one specific date (report_date).
# To verify this, we group March rows by (content_hash_id, report_date) pairs,
# and check whether any pair appears MORE THAN ONCE — which would mean the
# grain is violated (duplicate rows for the same page on the same day).
grain_check = con.sql(f"""
    SELECT content_hash_id, report_date, COUNT(*) AS n
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
    GROUP BY content_hash_id, report_date
    HAVING COUNT(*) > 1
""").df()

# If grain truly holds, this should be an EMPTY dataframe (zero duplicate pairs).
print("Rows violating grain (should be 0 if grain holds):", len(grain_check))
#What this does, in plain terms:
#it groups all March rows by (page, date) pairs, counts how many rows exist for each pair, and only keeps groups where the count is more than 1 (meaning a duplicate — same page, same day, appearing twice, which shouldn't happen if the grain is truly one-row-per-page-per-day). If the grain is clean, this query should return zero rows — an empty result is your proof.

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows violating grain (should be 0 if grain holds): 0


In [11]:
# Claim: our March slice has a specific number of rows, spanning exactly
# the March 2026 date range (no stray dates outside the window).
span_check = con.sql(f"""
    SELECT
        COUNT(*)              AS row_count,
        MIN(report_date)      AS earliest_date,
        MAX(report_date)      AS latest_date,
        COUNT(DISTINCT content_hash_id) AS unique_pages
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
""").df()

span_check
#COUNT(*) = total number of rows in your March slice (same idea as len(df) in pandas)
#MIN(report_date) / MAX(report_date) = the earliest and latest date actually present — proves your slice really is confined to March, nothing leaking in from other months
#COUNT(DISTINCT content_hash_id) = how many unique pages appear in March (useful extra context — tells you how many pages actually have March data, out of the ~427K–519K total pages in the warehouse)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,earliest_date,latest_date,unique_pages
0,9841378,2026-03-01,2026-03-31,331437


In [12]:
# Check what columns actually exist in the fact table, to find the right boolean flag
con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_daily']} LIMIT 1").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [13]:
# Claim: not all March rows have genuinely available GA4 data — some rows
# are zero-filled placeholders for clients without GA4 access or before
# their tracking started. Filtering with IS TRUE proves how much of the
# 9,841,378-row March slice can actually be trusted for GA4-based features.
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_march_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS rows_with_real_ga4,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS rows_with_real_gsc
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
""").df()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_march_rows,rows_with_real_ga4,rows_with_real_gsc
0,9841378,413966,3611061


In [14]:
# ============================================================
# FEATURE 1: February total GSC impressions, per page
# ------------------------------------------------------------
# Context: In the starter CSV, "impressions_90d" was already a single
# pre-aggregated number per row (one page = one number, summarizing its
# trailing 90 days). The CSV never had daily granularity.
#
# In the warehouse fact table, gsc_impressions is genuinely ONE ROW PER
# DAY PER PAGE — so to get something comparable to the old impressions_90d,
# we must SUM all of a page's daily gsc_impressions values across February,
# grouped by content_hash_id.
#
# No JOIN needed here — a join is only required when combining data from
# TWO DIFFERENT tables (e.g. fact_daily + dim_content for word_count).
# Since this feature only uses columns already inside fact_daily, a
# GROUP BY alone is sufficient.
#
# Available when: knowable by end of February — uses no March data,
# so it's safe as a feature for predicting a March-based label (Contract Q5).
#
# IMPORTANT: we filter on gsc_data_available IS TRUE, learned from Query 3 —
# otherwise we'd be summing in zero-filled placeholder rows from clients
# without real GSC tracking, which would understate true impressions.
# ============================================================
feb_impressions = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(gsc_impressions) AS feb_impressions
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-02-01' AND report_date < '2026-03-01'
      AND gsc_data_available IS TRUE
    GROUP BY content_hash_id
""").df()

feb_impressions.sort_values("feb_impressions", ascending=False).head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,feb_impressions
19538,content_8e1334d6356668e3,203401.0
14605,content_9c057b66c30a3abb,195648.0
96364,content_fec55986a1868d62,193954.0
15308,content_512dbad65bd5ade9,167303.0
19903,content_e241d6415ac9e534,164152.0
111877,content_e8a52cf3d5988c07,162129.0
116431,content_b99ea6861864dea5,160699.0
86979,content_f107e54b10b43725,156163.0
95494,content_e7b5dd4dff461ad2,154502.0
39570,content_acbcc847f8996314,148256.0


In [15]:
# ============================================================
# FEATURE 2: February click-through rate (CTR), per page
# ------------------------------------------------------------
# CTR = clicks / impressions -> what fraction of people who SAW the page
# (impressions) actually CLICKED on it. This tells us not just visibility,
# but how compelling the listing is.
#
# We reuse the same February window and the same gsc_data_available filter
# as Feature 1, for the same reason: rows without real GSC tracking are
# zero-filled placeholders, and including them would distort the ratio.
#
# We SUM clicks and impressions separately first (per page), THEN divide
# the two sums -- this is safer than averaging a daily CTR column directly,
# because summing raw counts first avoids letting low-traffic days (where
# a tiny number of clicks/impressions can create a wildly extreme ratio)
# unfairly dominate the average.
#
# Available when: knowable by end of February -- no March data used,
# so it's safe as a feature for predicting a March-based label.
# ============================================================
feb_ctr = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(gsc_clicks) AS feb_clicks,
        SUM(gsc_impressions) AS feb_impressions,
        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN CAST(SUM(gsc_clicks) AS DOUBLE) / SUM(gsc_impressions)
            ELSE NULL
        END AS feb_ctr
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-02-01' AND report_date < '2026-03-01'
      AND gsc_data_available IS TRUE
    GROUP BY content_hash_id
""").df()

feb_ctr.sort_values("feb_impressions", ascending=False).head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,feb_clicks,feb_impressions,feb_ctr
19538,content_8e1334d6356668e3,2.0,203401.0,0.000010
14605,content_9c057b66c30a3abb,1.0,195648.0,0.000005
96364,content_fec55986a1868d62,0.0,193954.0,0.000000
15308,content_512dbad65bd5ade9,3310.0,167303.0,0.019784
19903,content_e241d6415ac9e534,401.0,164152.0,0.002443
119593,content_e8a52cf3d5988c07,627.0,162129.0,0.003867
114342,content_b99ea6861864dea5,273.0,160699.0,0.001699
86979,content_f107e54b10b43725,883.0,156163.0,0.005654
95494,content_e7b5dd4dff461ad2,2508.0,154502.0,0.016233
37137,content_acbcc847f8996314,239.0,148256.0,0.001612


In [16]:
# ============================================================
# FEATURE 3: February average search position, per page
# ------------------------------------------------------------
# gsc_avg_position tells us roughly where a page ranks in search results
# (lower number = better rank, e.g. position 1 = top result).
#
# KNOWN GOTCHA (confirmed earlier in the starter CSV, w01): a value of 0
# does NOT mean "ranked at position zero" -- it's a sentinel for "no
# position data." If we naively AVG() this column including those 0-rows,
# we'd artificially drag the average toward a falsely "better" position.
#
# Fix: filter out rows where gsc_avg_position = 0 (or where the day had no
# real position data) BEFORE averaging, same as we did with the CSV's
# avg_position column. We also keep the gsc_data_available filter for the
# same reason as Features 1 and 2.
#
# Available when: knowable by end of February -- no March data used.
# ============================================================
feb_position = con.sql(f"""
    SELECT
        content_hash_id,
        AVG(gsc_avg_position) AS feb_avg_position
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-02-01' AND report_date < '2026-03-01'
      AND gsc_data_available IS TRUE
      AND gsc_avg_position > 0
    GROUP BY content_hash_id
""").df()

feb_position.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,feb_avg_position
0,content_36bee0a093d0711d,5.500000
1,content_1546aabff77c05a4,5.000000
2,content_cae1d5374958a649,6.219121
3,content_dd66eecf9626cab8,6.407819
4,content_c51f1e8ef5502177,7.541667
5,content_0674cc4ae0f68a90,8.063910
6,content_0bbad1e88286bfc6,4.000000
7,content_388b1fcec3596848,5.401852
8,content_0ffc84b3be2f3bc1,9.690476
9,content_456ab2db28595187,4.330032


In [17]:
# ============================================================
# FEATURE 4: Word count, per page (static, from dim_content)
# ------------------------------------------------------------
# Unlike Features 1-3, this comes from dim_content, not fact_daily --
# it's a STATIC characteristic of the page itself, not a daily-changing
# traffic number. It doesn't vary day-to-day for a single page, but it
# DOES vary across different pages -- e.g. short pages (low word_count)
# might be more prone to decline than long, thorough ones. That
# across-page variation is what gives the model real signal to learn from.
#
# No GSC/GA4 availability issue here -- dim_content isn't tied to daily
# tracking, so this is present for essentially every page.
#
# Available when: word_count is set whenever the page is created/updated
# -- as long as we only use its value AS OF a point before March (and the
# page isn't being actively rewritten during our prediction window), this
# is safe. Assumption stated openly: we assume word_count didn't change
# meaningfully between whenever it was last set and March.
# ============================================================
content_features = con.sql(f"""
    SELECT
        content_hash_id,
        word_count
    FROM {TABLES['dim_content']}
""").df()

content_features.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,word_count
0,content_004de9653278b5a4,2555
1,content_00dc5efae381b2ab,2430
2,content_01410f2556c327ac,2645
3,content_019f27f634053ca7,2522
4,content_01efa71faea45dcc,2552
5,content_01fc9e2e57898b55,2672
6,content_0212158fa61c5fcb,2396
7,content_023d807c7d922db1,1929
8,content_026f7405cc253242,2481
9,content_02c8b23ea5bdb275,2357


In [18]:
# ============================================================
# FEATURE 5: Search volume for the page's target keyword (static, from dim_content)
# ------------------------------------------------------------
# search_volume measures keyword DEMAND -- how often people search for
# this term overall -- which is different from ctr (how well THIS page
# converts the traffic it gets) or impressions (how much traffic THIS
# page actually received). A page could have low impressions/CTR simply
# because it targets a low-demand keyword, not because the page itself
# is weak -- search_volume helps separate those two explanations.
#
# Static, from dim_content -- same reasoning as Feature 4: doesn't vary
# day-to-day for one page, but varies meaningfully across pages, giving
# the model real signal to learn from.
#
# Available when: search_volume is a property of the target keyword,
# knowable independent of March's actual performance -- safe under Q5.
# ============================================================
content_features_v2 = con.sql(f"""
    SELECT
        content_hash_id,
        word_count,
        search_volume
    FROM {TABLES['dim_content']}
""").df()

content_features_v2.head(10)

,content_hash_id,word_count,search_volume
0,content_004de9653278b5a4,2555,30
1,content_00dc5efae381b2ab,2430,10
2,content_01410f2556c327ac,2645,480
3,content_019f27f634053ca7,2522,0
4,content_01efa71faea45dcc,2552,2400
5,content_01fc9e2e57898b55,2672,260
6,content_0212158fa61c5fcb,2396,10
7,content_023d807c7d922db1,1929,10
8,content_026f7405cc253242,2481,4400
9,content_02c8b23ea5bdb275,2357,390


In [19]:
# ============================================================
# Combine all 5 features into a single per-page feature table.
# Features 1-3 come from fact_daily (February, aggregated per page).
# Features 4-5 come from dim_content (static page characteristics).
# We join everything on content_hash_id.
# Using LEFT JOIN from content_features_v2 as the base, since dim_content
# should have the most complete page coverage.
# ============================================================
feature_frame = content_features_v2 \
    .merge(feb_impressions, on="content_hash_id", how="left") \
    .merge(feb_ctr[["content_hash_id", "feb_ctr"]], on="content_hash_id", how="left") \
    .merge(feb_position, on="content_hash_id", how="left")

print(feature_frame.shape)
feature_frame.head(10) #we've only printed 10 thats why there are many NANs

(519606, 6)


,content_hash_id,word_count,search_volume,feb_impressions,feb_ctr,feb_avg_position
0,content_004de9653278b5a4,2555,30,NaN,NaN,NaN
1,content_00dc5efae381b2ab,2430,10,NaN,NaN,NaN
2,content_01410f2556c327ac,2645,480,NaN,NaN,NaN
3,content_019f27f634053ca7,2522,0,NaN,NaN,NaN
4,content_01efa71faea45dcc,2552,2400,NaN,NaN,NaN
5,content_01fc9e2e57898b55,2672,260,NaN,NaN,NaN
6,content_0212158fa61c5fcb,2396,10,NaN,NaN,NaN
7,content_023d807c7d922db1,1929,10,NaN,NaN,NaN
8,content_026f7405cc253242,2481,4400,NaN,NaN,NaN
9,content_02c8b23ea5bdb275,2357,390,NaN,NaN,NaN


In [20]:
print("Pages in dim_content:", len(content_features_v2))
print("Pages in feb_impressions:", len(feb_impressions))
print("Rows with NaN feb_impressions in final table:", feature_frame["feb_impressions"].isna().sum())
# Note: feature_frame currently mixes pages WITH real February GSC data
# and pages WITHOUT (NaN). Before training any model, we must drop rows
# with missing features -- otherwise the model can't process them.
# This means our model only covers "pages with real February tracking data,"
# not the full page inventory -- an honest limitation to state later.

Pages in dim_content: 519606
Pages in feb_impressions: 153559
Rows with NaN feb_impressions in final table: 366047


In [21]:
# ============================================================
# WHY 0.8 (i.e. a 20% drop) AS THE DECLINE THRESHOLD
# ------------------------------------------------------------
# march_impressions < 0.8 * feb_impressions
# means: March impressions fell below 80% of February's -- i.e. the page
# lost more than 20% of its traffic month-over-month.
#
# This threshold is a POLICY CHOICE, not a universal rule (same idea as
# the "180 days = stale" cutoff chosen in w01) -- a stricter threshold
# (e.g. 0.9) would flag smaller dips as "declining"; a looser one (e.g.
# 0.7) would only flag severe drops. We adopt 0.8, matching FlyRank's own
# reference notebook, since a 20% month-over-month drop is a meaningful,
# deliberate signal rather than ordinary noise.
# ============================================================
# Same logic as Feature 1 (February impressions), just shifted to March.
# We need march_impressions to BUILD the label -- this is expected and
# required, not a leak by itself (see earlier comment on what the leak
# actually is).
march_impressions = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
      AND gsc_data_available IS TRUE
    GROUP BY content_hash_id
""").df()

march_impressions.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,march_impressions
0,content_b7e512995f79d5a6,1140.0
1,content_05597932fe4da067,57.0
2,content_905aa32a0230694e,149.0
3,content_05434271b257bb68,1421.0
4,content_d056587ff7faca0c,2770.0


In [22]:
# Merge Feb and March impressions into one table, keyed by content_hash_id.
# Using INNER merge here (not left) -- we only want pages that have REAL
# data in BOTH months, since comparing Feb vs March only makes sense if
# both numbers are genuinely measured, not one real and one missing.
trend_data = feb_impressions.merge(march_impressions, on="content_hash_id", how="inner")

# Build the label: did the page lose more than 20% of its traffic
# from February to March? (see threshold explanation above)
trend_data["is_declining"] = (
    trend_data["march_impressions"] < 0.8 * trend_data["feb_impressions"]
).astype(int)

print(trend_data.shape)
print("Decline rate:", trend_data["is_declining"].mean())
trend_data.head(10)

(134238, 4)
Decline rate: 0.19983164230694736


,content_hash_id,feb_impressions,march_impressions,is_declining
0,content_1eea820697c3b95a,299.0,315.0,0
1,content_9abd8b303f805847,733.0,14536.0,0
2,content_5f58c55cbfee172a,514.0,387.0,1
3,content_6fe390ba3af1e456,2931.0,4697.0,0
4,content_3ad5d2160242b9ca,970.0,1004.0,0
5,content_a2bd730a7cf68316,551.0,551.0,0
6,content_cbe43d4b6ce2d320,291.0,344.0,0
7,content_babd931911c9ee33,2680.0,4946.0,0
8,content_9c36ace83c73b5eb,416.0,302.0,1
9,content_431784c057b25a5d,3641.0,1871.0,1


In [23]:
print(trend_data["is_declining"].mean())
# ============================================================
# BUILDING THE LABEL: February vs March comparison
# ------------------------------------------------------------
# We merge feb_impressions and march_impressions into one table so both
# numbers sit in the same row, per page -- required before any comparison
# is possible.
#
# INNER merge (not left): we only keep pages with REAL data in BOTH
# months, since comparing Feb vs March only makes sense if both numbers
# are genuinely measured (not one real, one missing/NaN).
#
# is_declining = 1 if march_impressions < 0.8 * feb_impressions
#              (i.e. March lost more than 20% of February's traffic)
#              else 0
#
# This is a BOOLEAN-style label (only ever 0 or 1), same idea as the
# CSV's is_declining_label -- but built here from two real calendar
# months we verified ourselves, rather than loaded from a pre-made
# column.
#
# Result: 19.98% decline rate under this definition -- different from
# the CSV's 54.2%, because this uses a stricter, self-defined threshold
# (a genuine 20%+ month-over-month drop), not the CSV's original
# trend_direction formula. Different definition -> different number,
# not an error.
#
# IMPORTANT: no model or training has happened yet at this step -- this
# is pure arithmetic on two real, already-measured numbers. Leakage is
# a concept that applies later, once we start choosing which columns to
# feed a MODEL as features.
# ============================================================

0.19983164230694736


what we have till now:
Feb impressions ✅ (Feature 1)
Feb CTR ✅ (Feature 2)
Feb avg position ✅ (Feature 3)
word_count ✅ (Feature 4, static)
search_volume ✅ (Feature 5, static)
March impressions ✅ (just now — needed only to build the label, NOT one of your 5 features)
The label itself (is_declining) ✅ — just built, 19.98% decline rate confirmed

In [24]:
# Combine our 5 honest features with the label, all keyed by content_hash_id
training_table = feature_frame.merge(
    trend_data[["content_hash_id", "is_declining"]],
    on="content_hash_id",
    how="inner"  # only keep pages where we have BOTH features and a label
)

print("Before dropping NaNs:", training_table.shape)

feature_cols = ["word_count", "search_volume", "feb_impressions", "feb_ctr", "feb_avg_position"]
training_table_clean = training_table.dropna(subset=feature_cols)

print("After dropping NaNs:", training_table_clean.shape)
print("Decline rate in final training set:", training_table_clean["is_declining"].mean())
training_table_clean.head()

Before dropping NaNs: (134238, 7)
After dropping NaNs: (75189, 7)
Decline rate in final training set: 0.15937171660748248


,content_hash_id,word_count,search_volume,feb_impressions,feb_ctr,feb_avg_position,is_declining
0,content_04c67f3541177192,3168,0,246.0,0.004065,19.272721,0
1,content_05acc92c165f4386,4135,20,137.0,0.000000,9.754703,1
2,content_0f30e04e709c7b5d,3211,0,121.0,0.000000,8.994742,0
3,content_1207efddce873942,3465,0,174.0,0.000000,13.195257,0
4,content_167472cd0802a8f3,3149,0,164.0,0.000000,11.451102,0


In [25]:
#training honest model (no leak, 5 features)
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

feature_cols = ["word_count", "search_volume", "feb_impressions", "feb_ctr", "feb_avg_position"]
X = training_table_clean[feature_cols]
y = training_table_clean["is_declining"]

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

model_honest = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

print("Base rate (always predict majority):", max(y_te.mean(), 1 - y_te.mean()))
print(classification_report(y_te, model_honest.predict(X_te), digits=3))

Base rate (always predict majority): 0.8406213426960315
              precision    recall  f1-score   support

           0      0.849     0.977     0.909     15802
           1      0.416     0.085     0.142      2996

    accuracy                          0.835     18798
   macro avg      0.633     0.531     0.525     18798
weighted avg      0.780     0.835     0.787     18798



In [ ]:
# ============================================================
# HONEST BASELINE MODEL (5 real features, no leakage)
# ------------------------------------------------------------
# Base rate (always guess "not declining"): 0.8406
# -- since 84.1% of pages in our clean training set are NOT declining,
# a model that learns NOTHING and just always guesses the majority class
# would already be "right" 84.1% of the time. This is the honest floor
# any real model must clearly beat to prove it learned something useful.
#
# Our actual model's accuracy: 0.835 -- essentially TIED with the dumb
# baseline (0.8406), meaning raw accuracy barely tells us anything here.
#
# The real story is in the per-class numbers: for class 1 (declining,
# the class we actually care about), RECALL is only 0.085 -- the model
# catches just 8.5% of genuinely declining pages. It's mostly defaulting
# to "not declining" for nearly everyone, which is how it fakes a
# reasonable-looking accuracy while being nearly useless at its actual job.
#
# This is our honest number to compare against the deliberate leak below.
# ============================================================

In [26]:
# ============================================================
# THE DELIBERATE LEAK: adding march_impressions as a "feature"
# ------------------------------------------------------------
# march_impressions is a direct ingredient of our label formula:
#   is_declining = (march_impressions < 0.8 * feb_impressions)
# Feeding it in as a feature lets the model just re-derive the label
# almost perfectly, instead of learning any real pattern -- the same
# trap as notebook 02's trend_pct experiment.
# ============================================================

training_table_leaky = training_table.merge(
    march_impressions, on="content_hash_id", how="inner"
)
training_table_leaky_clean = training_table_leaky.dropna(
    subset=feature_cols + ["march_impressions"]
)

feature_cols_leaky = feature_cols + ["march_impressions"]
X_leak = training_table_leaky_clean[feature_cols_leaky]
y_leak = training_table_leaky_clean["is_declining"]

X_tr_l, X_te_l, y_tr_l, y_te_l = train_test_split(
    X_leak, y_leak, test_size=0.25, random_state=42, stratify=y_leak
)

model_leaky = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr_l, y_tr_l)

print("=== LEAKY MODEL (includes march_impressions) ===")
print(classification_report(y_te_l, model_leaky.predict(X_te_l), digits=3))

=== LEAKY MODEL (includes march_impressions) ===
              precision    recall  f1-score   support

           0      0.982     0.999     0.990     15802
           1      0.993     0.904     0.946      2996

    accuracy                          0.984     18798
   macro avg      0.987     0.951     0.968     18798
weighted avg      0.984     0.984     0.983     18798



In [27]:
# ============================================================
# REMOVING THE LEAK
# ------------------------------------------------------------
# LEAKY model (includes march_impressions): accuracy 0.984, class-1 recall 0.904
# -- looks "amazing," but this is fake: march_impressions is a direct
# ingredient of the label formula, so the model is just reading the
# answer, not learning a real pattern (same trap as notebook 02's
# trend_pct experiment).
#
# HONEST model (5 real features only, no march data): accuracy 0.835,
# class-1 recall 0.085 -- barely beats the dumb "always guess majority"
# baseline (0.8406). This is the real, trustworthy number.
#
# We discard march_impressions as a feature and keep the honest model
# as our actual result going forward.
# ============================================================
print("HONEST model (kept):")
print(classification_report(y_te, model_honest.predict(X_te), digits=3))

print("\nLEAKY model (discarded, shown only to demonstrate the trap):")
print(classification_report(y_te_l, model_leaky.predict(X_te_l), digits=3))

HONEST model (kept):
              precision    recall  f1-score   support

           0      0.849     0.977     0.909     15802
           1      0.416     0.085     0.142      2996

    accuracy                          0.835     18798
   macro avg      0.633     0.531     0.525     18798
weighted avg      0.780     0.835     0.787     18798


LEAKY model (discarded, shown only to demonstrate the trap):
              precision    recall  f1-score   support

           0      0.982     0.999     0.990     15802
           1      0.993     0.904     0.946      2996

    accuracy                          0.984     18798
   macro avg      0.987     0.951     0.968     18798
weighted avg      0.984     0.984     0.983     18798



## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

(I ask claude to ask me questions to get to the answer and it then refines or corrects my answers as needed. So the text below and other texts are generated by claude but have my content and ideas)

This starter dataset has several limits that shape what we can honestly claim:

Unbalanced/unknown history: This is a single 30,000-row anonymized slice. We don't know how many distinct clients it represents or how their history depth varies (the full warehouse has 104 clients with wildly different tracking start dates). Findings here should not be generalized to "all FlyRank clients" without checking the fuller warehouse data.

No calendar dates, only relative windows: The CSV has no explicit snapshot or report date — only relative windows (_90d, last_30d/prev_30d, days_since_last_update). This means we cannot verify whether a future "predict decline in the next 30 days" model's feature window and target window would truly avoid overlapping in time; we'd need explicit dates (available in the warehouse's report_date column) to guarantee that safely.

Sparse AI-traffic data: 93.6% of rows have ai_traffic_pct = 0. We can say most of the traffic in this dataset isn't AI-driven, but we cannot confidently claim individual pages "get no AI traffic" — the signal is simply too thin to support page-level claims.

Confounded/missing provider data: provider_used/model_used are 71.5% missing, and the missingness itself may not be random (e.g., older pages may predate certain AI tools entirely). Any comparison across AI models would likely reflect when content was made, not a genuine causal effect of the model used — so this field was excluded rather than used to support any claim.

Sentinel values limit coverage: 1,205 rows (4%) have avg_position = 0, meaning no ranking data — not a real rank of zero. Any analysis using avg_position must exclude these rows, meaning conclusions apply only to "pages with ranking data," not the full page inventory.

Correlation, not causation: Everything in this dataset is observational. We can find that certain signals (staleness, low CTR, etc.) correlate with decline, but we cannot claim any of them cause decline, and we cannot claim that reviewing a flagged page will cause it to recover — that would require an experiment or causal design, which this dataset doesn't support.

Named limitation: Our clean, leakage-free training set (75,189 rows) covers only a small fraction of the ~519,606 pages in dim_content — most were excluded because they lacked genuine GSC tracking in both February and March (gsc_data_available IS TRUE filtered out a large majority; recall only 36.7% of March rows had real GSC data). Any conclusions from this analysis apply only to "pages with available GSC tracking in both months," not FlyRank's full page inventory.

Separately, our honest model (0.835 accuracy) barely outperformed a dumb "always guess majority" baseline (0.841), with very low recall (0.085) on the declining class. We can't be fully certain why performance was this weak — it's plausible that (1) five features and a two-month comparison genuinely don't carry enough signal for this problem, (2) stronger signals exist elsewhere in the warehouse (e.g. query-diversity data from fact_content_query_90d) that we deliberately left out to stay within the 5-feature limit, or (3) a different model or better-tuned hyperparameters could extract more from the same features. We don't have enough evidence yet to isolate which explanation dominates — a fuller investigation with more features and model comparisons (as future weeks will build toward) would be needed to distinguish between them.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.